In [32]:
# Imports
import os
import joblib
import numpy as np
import pandas as pd

# Used functions

1. Utility function to load `.xlsx` files from a directory, optionally using custom keys, and return a dictionary of concatenated DataFrames.

In [39]:
# 1.
def load_data_to_dict(
    dir_path: str, cache_path: str = "loaded_files_cache.pkl", **file_keys: str
) -> dict[str, pd.DataFrame]:
    """
    Loads .xlsx files from the specified directory, concatenates all sheets
    into a single DataFrame per file, and caches results to avoid reloading.

    Parameters
    ----------
    dir_path : str
        Path to the directory containing .xlsx files.
    cache_path : str
        Path to the joblib file where the cache will be saved/loaded.
    **file_keys : str, optional
        Mapping of custom keys to filenames (e.g., train="train_data.xlsx").

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary with keys as specified (or derived from filenames)
        and values as concatenated DataFrames.
    Raises
    ------
    FileNotFoundError
        If the specified directory does not exist.
    ValueError
        If the directory is empty or no valid .xlsx files were loaded.

    Example:
    -------
    >>> data = load_data_to_dict("data/", train="train_data.xlsx")
    >>> df = data["train"]
    """
    # Load cache if it exists
    if os.path.exists(cache_path):
        loaded_files_cache = joblib.load(cache_path)
    else:
        loaded_files_cache = {}

    data = {}

    if not os.path.exists(dir_path):
        raise FileNotFoundError(f"Directory '{dir_path}' not found!")
    if not os.listdir(dir_path):
        raise ValueError(f"Directory '{dir_path}' is empty!")

    for key, filename in file_keys.items():
        if key in loaded_files_cache:
            print(f"[Skip] '{filename}' already loaded as '{key}'")
            data[key] = loaded_files_cache[key]
            continue

        full_path = os.path.join(dir_path, filename)
        file_key = next(
            (k for k, v in file_keys.items() if v == filename),
            os.path.splitext(filename)[0],
        )

        try:
            xls = pd.ExcelFile(full_path)
            dfs = [xls.parse(sheet_name) for sheet_name in xls.sheet_names]
            combined_df = pd.concat(dfs, ignore_index=True)
            data[file_key] = combined_df
            loaded_files_cache[file_key] = combined_df
            print(f"Loaded and combined '{filename}' as '{file_key}'")
        except Exception as e:
            print(f"Error loading file '{filename}': {str(e)}")
            continue

    # Save updated cache
    joblib.dump(loaded_files_cache, cache_path)

    if not data:
        raise ValueError("No valid .xlsx files were loaded!")

    return data

2. Computes the `SoH` for each row in a DataFrame as the ratio of `discharge_capacity` to `nominal_capacity`:

$$
\textit{SoH} = \frac{\text{discharge\_capacity}}{\text{nominal\_capacity}}
$$

In [29]:
# 2.
def get_soh(
    data: dict[str, pd.DataFrame], key: str, nominal_capacity: float = None
) -> pd.Series:
    """
    Computes State of Health (SoH) for a given DataFrame in a dictionary.

    Parameters:
    -----------
    data : dict[str, pd.DataFrame]
        Dictionary of DataFrames where keys are sample names.
    key : str
        Key of the DataFrame to compute SoH for.
    nominal_capacity : float, optional
        Nominal or reference capacity (e.g., from initial measurement or spec sheet).
        Can be passed as a variable or value. If not provided, raises an error.

    Returns:
    --------
    pd.Series
        Series of SoH values computed as Discharge_Capacity / nominal_capacity.

    Raises
    ------
    ValueError
        - If nominal_capacity is not provided
        - If input DataFrame lacks 'Discharge_Capacity(Ah)' column
    """
    if nominal_capacity is None:
        raise ValueError("nominal_capacity must be provided as a number or variable.")

    df = data[key]

    if "Discharge_Capacity(Ah)" not in df.columns:
        raise ValueError(f"'Discharge_Capacity(Ah)' column not found in '{key}'.")

    soh_series = df["Discharge_Capacity(Ah)"] / nominal_capacity
    soh_series.name = "State_of_Health"

    return soh_series

# Data preprocessing

## Extract data from the incremental current OCV test

### Initialization

Load `.xlsx` files using the `load_data_to_dict` function from **Incremental Current OCV tests** for Sample 1 and Sample 2 to a dictionary-based structure, where each key corresponds to a specific test condition (initial, 0°C, 25°C, 45°C).

In [40]:
# Sample1
incremental_current_OCV_test_sp1 = load_data_to_dict(
    "../data/OCV-SOC/Incremental_Current_OCV/Sample1",
    initial="Incremental_current_OCV_test_Initial_capacity_SP1.xlsx",
    temp_0="Incremental_current_OCV_test_SP1-0.xlsx",
    temp_25="Incremental_current_OCV_test_SP1-25.xlsx",
    temp_45="Incremental_current_OCV_test_SP1-45.xlsx",
)

[Skip] 'Incremental_current_OCV_test_Initial_capacity_SP1.xlsx' already loaded as 'initial'
[Skip] 'Incremental_current_OCV_test_SP1-0.xlsx' already loaded as 'temp_0'
[Skip] 'Incremental_current_OCV_test_SP1-25.xlsx' already loaded as 'temp_25'
[Skip] 'Incremental_current_OCV_test_SP1-45.xlsx' already loaded as 'temp_45'


In [42]:
# Sample2
incremental_current_OCV_test_sp2 = load_data_to_dict(
    "../data/OCV-SOC/Incremental_Current_OCV/Sample2",
    initial="Incremental_current_OCV_test_Initial_capacity_SP2.xlsx",
    temp_0="Incremental_current_OCV_test_SP2-0.xlsx",
    temp_25="Incremental_current_OCV_test_SP2-25.xlsx",
    temp_45="Incremental_current_OCV_test_SP2-45.xlsx",
)

[Skip] 'Incremental_current_OCV_test_Initial_capacity_SP2.xlsx' already loaded as 'initial'
[Skip] 'Incremental_current_OCV_test_SP2-0.xlsx' already loaded as 'temp_0'
[Skip] 'Incremental_current_OCV_test_SP2-25.xlsx' already loaded as 'temp_25'
[Skip] 'Incremental_current_OCV_test_SP2-45.xlsx' already loaded as 'temp_45'


### Generating the Target Variable

Extract the nominal capacity for each sample based on the maximum `discharge_capacity` recorded during the initial test. 
This value will later be used as a reference for calculating the `SoH`.

In [46]:
# Sample1 nominal_capacity
nominal_capacity_sp1 = incremental_current_OCV_test_sp1["initial"][
    "Discharge_Capacity(Ah)"
].max()

In [47]:
# Sample2 nominal_capacity
nominal_capacity_sp2 = incremental_current_OCV_test_sp2["initial"][
    "Discharge_Capacity(Ah)"
].max()

Remove rows with zero `discharge_capacity` from all dataframes to avoid invalid SoH values (e.g., zero or undefined). 
This ensures that the target variable is computed only from valid discharge segments.

In [49]:
# Sample1 cleaning
for key, df in incremental_current_OCV_test_sp1.items():
    if "Discharge_Capacity(Ah)" in df.columns:
        cleaned_df = df.loc[df["Discharge_Capacity(Ah)"] > 0].copy()
        incremental_current_OCV_test_sp1[key] = cleaned_df

In [50]:
# Sample2 cleaning
for key, df in incremental_current_OCV_test_sp2.items():
    if "Discharge_Capacity(Ah)" in df.columns:
        cleaned_df = df.loc[df["Discharge_Capacity(Ah)"] > 0].copy()
        incremental_current_OCV_test_sp2[key] = cleaned_df

Define the target variable `SoH` as the ratio of current `discharge_capacity` to the `nominal_capacity` using function `get_soh` for each sample and test condition

In [52]:
# Sample1 target
soh_target_value_OCV_test_sp1 = {
    "target_0": get_soh(
        incremental_current_OCV_test_sp1,
        "temp_0",
        nominal_capacity=nominal_capacity_sp1,
    ),
    "target_25": get_soh(
        incremental_current_OCV_test_sp1,
        "temp_25",
        nominal_capacity=nominal_capacity_sp1,
    ),
    "target_45": get_soh(
        incremental_current_OCV_test_sp1,
        "temp_45",
        nominal_capacity=nominal_capacity_sp1,
    ),
}

In [53]:
# Sample2 target
soh_target_value_OCV_test_sp2 = {
    "target_0": get_soh(
        incremental_current_OCV_test_sp2,
        "temp_0",
        nominal_capacity=nominal_capacity_sp2,
    ),
    "target_25": get_soh(
        incremental_current_OCV_test_sp2,
        "temp_25",
        nominal_capacity=nominal_capacity_sp2,
    ),
    "target_45": get_soh(
        incremental_current_OCV_test_sp2,
        "temp_45",
        nominal_capacity=nominal_capacity_sp2,
    ),
}

### Cleaning & Parsing

In [12]:
incremental_current_OCV_test_sp1["initial"].head()

,Data_Point,Test_Time(s),Date_Time,Step_Time(s),Step_Index,Cycle_Index,Current(A),Voltage(V),Charge_Capacity(Ah),Discharge_Capacity(Ah),Charge_Energy(Wh),Discharge_Energy(Wh),dV/dt(V/s),Internal_Resistance(Ohm),Is_FC_Data,AC_Impedance(Ohm),ACI_Phase_Angle(Deg)
12980,12981,13113.561566,2015-10-16 13:12:52,1.015476,6,1,-0.999981,4.066597,1.996313,0.000282,7.798005,0.001147,-0.000713,0.123002,0,0,0
12981,12982,13114.561654,2015-10-16 13:12:53,2.015564,6,1,-1.000162,4.065139,1.996313,0.000560,7.798005,0.002277,-0.001005,0.123002,0,0,0
12982,12983,13115.577322,2015-10-16 13:12:54,3.031232,6,1,-1.000343,4.064005,1.996313,0.000842,7.798005,0.003424,-0.001231,0.123002,0,0,0
12983,12984,13116.577390,2015-10-16 13:12:55,4.031300,6,1,-1.000162,4.063195,1.996313,0.001120,7.798005,0.004553,-0.001393,0.123002,0,0,0
12984,12985,13117.592102,2015-10-16 13:12:56,5.046012,6,1,-1.000162,4.062223,1.996313,0.001398,7.798005,0.005682,-0.001005,0.123002,0,0,0
